# 02. 주소 → 우편번호 조회 색인 v1 (선택 지역)

**목적**: 사이트가 서버 없이 주소로 우편번호를 찾을 수 있게, 우편번호 DB를 **시군구별 정적 JSON 조각**으로 만든다.
브라우저가 사용자가 입력한 시군구의 조각 하나만 내려받아 그 안에서 찾는다.

**입력** (`data/raw/`, 저장소에는 올리지 않음): `zipcode_DB/`(건물 단위), `rangeaddr/`(도로명·지번 범위, 참고자료)

**출력** (`data/processed/addr/`)
- `names.json` : 시군구 목록과 도로명·읍면동 이름 → 시군구 색인 (입력에서 시군구를 알아내는 용도)
- `{시군구코드}.json` : 그 시군구의 도로명 주소(건물), 건물명, 지번 주소, 도로명·지번 범위
- `meta.json`, `../qa/address_index_qa_v1.xlsx`

**조회 정확도 등급 (사이트에 그대로 표시)**
1. `건물 일치` : 도로명(또는 지번)과 번호가 우편번호 DB의 건물과 정확히 같다
2. `범위 기준` : 건물은 없지만 우체국 범위주소(참고자료)의 범위 안에 들어간다. 범위는 참고용이다

**확인 방법**: 마지막 절에서 DB의 모든 건물을 이 색인으로 다시 조회해 원래 우편번호가 나오는지 전수 대조한다.

In [1]:
import os, json, re, collections, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)

RAW = Path(os.environ.get("KRPC_RAW", Path.cwd().parent / "raw"))
OUT = Path(os.environ.get("KRPC_OUT", Path.cwd().parent / "processed"))
ADDR = OUT / "addr"
ADDR.mkdir(parents=True, exist_ok=True)
for old in ADDR.glob("*.json"):        # 이전 실행 잔여물 제거 (지역을 바꿔 다시 돌릴 때 대비)
    old.unlink()

REGIONS = [
    {"key": "sejong",    "sido": "세종특별자치시", "db": "세종특별자치시.txt", "sigungu_prefix": None},
    {"key": "daejeon",   "sido": "대전광역시",     "db": "대전광역시.txt",     "sigungu_prefix": None},
    {"key": "gyeonggi3", "sido": "경기도",         "db": "경기도.txt",         "sigungu_prefix": ["성남시", "수원시", "용인시"]},
]
BASE_DATE = "2026-10-08"
NOTICE_DATE = "2026-10-13"
print("RAW =", RAW.resolve()); print("OUT =", OUT.resolve())

RAW = /tmp/claude-0/-home-claude/f10cf2f9-ff9c-5654-bcf6-30294dc870a2/scratchpad/raw
OUT = /home/claude/kr-postcode/data/processed


## 1. 우편번호 DB 읽기
한 줄 = 건물 1건. 도로명 주소(도로명, 본번·부번, 지하 여부)와 지번 주소(법정동, 산 여부, 본번·부번)를 둘 다 쓴다.

In [2]:
DB_COLS = ["우편번호", "시도", "시군구", "읍면", "도로명", "지하여부", "건물번호본번", "건물번호부번",
           "시군구용건물명", "법정동코드", "법정동명", "리명", "산여부", "지번본번", "지번부번"]
frames = []
for r in REGIONS:
    d = pd.read_csv(RAW / "zipcode_DB" / r["db"], sep="|", encoding="utf-8-sig", dtype=str, usecols=DB_COLS, keep_default_na=False)
    if r["sigungu_prefix"]:
        d = d[d["시군구"].str.startswith(tuple(r["sigungu_prefix"]))]
    frames.append(d)
db = pd.concat(frames, ignore_index=True)
db["code"] = db["법정동코드"].str[:5]
db["emd"] = db["읍면"]
db["label"] = np.where(db["법정동명"] != "", db["법정동명"], (db["읍면"] + " " + db["리명"]).str.strip())
for c in ["건물번호본번", "건물번호부번", "지번본번", "지번부번", "지하여부", "산여부"]:
    assert db[c].str.fullmatch(r"\d+").all(), f"{c} 숫자가 아닌 값"
    db[c] = db[c].astype(int)
assert db["우편번호"].str.fullmatch(r"\d{5}").all()
print(f"{len(db):,d}행, 시군구 {db['code'].nunique()}개")

# 시군구 코드 ↔ 이름 (사이트의 hierarchy.json과 같은 이름을 쓴다)
hier = json.loads((OUT / "hierarchy.json").read_text(encoding="utf-8"))
CODE_NAME = {g["code"]: g["name"] for h in hier for g in h["sigungu"]}
CODE_SIDO = {g["code"]: h["sido"] for h in hier for g in h["sigungu"]}
assert set(db["code"]) == set(CODE_NAME), "hierarchy.json 과 코드 불일치"

# (시도, 시군구) → 코드 : 범위주소 파일에 코드가 없어 이 표로 연결한다. 세종은 시군구가 비어 있다.
key2code = db.groupby(["시도", "시군구"])["code"].agg(lambda s: sorted(set(s)))
assert (key2code.str.len() == 1).all(), "시도·시군구가 둘 이상의 코드에 걸림"
KEY2CODE = {k: v[0] for k, v in key2code.items()}
print(len(KEY2CODE), "개 (시도, 시군구) 키")

296,074행, 시군구 16개


16 개 (시도, 시군구) 키


## 2. 범위주소 읽기 (참고자료)
범위종류: 0 해당 주소 1건, 1 홀수 전체, 2 짝수 전체, 3 전체. 범위 안의 건물은 모두 같은 우편번호라는 뜻이다.

In [3]:
rg = pd.read_csv(RAW / "rangeaddr" / "20261013_도로명범위.txt", sep="|", encoding="utf-8-sig", dtype=str, keep_default_na=False)
rg.columns = ["zip", "sido", "sido_en", "sgg", "sgg_en", "emd", "emd_en", "road", "road_en", "under", "sb", "sbu", "eb", "ebu", "kind"]
rg_all = len(rg)
rg["code"] = [KEY2CODE.get(k) for k in zip(rg["sido"], rg["sgg"])]
rg = rg[rg["code"].notna()].copy()
# 끝번호가 빈 줄은 번호 1개만 가리키는 줄(범위종류 0 등)이라 끝=시작으로 둔다
rg_blank_end = int((rg["eb"] == "").sum())
rg["eb"] = np.where(rg["eb"] == "", rg["sb"], rg["eb"]); rg["ebu"] = np.where(rg["ebu"] == "", rg["sbu"], rg["ebu"])
print("도로명범위 끝번호 빈 줄", rg_blank_end, "/ 범위종류별", rg.loc[rg["kind"].astype(int) >= 0, "kind"].value_counts().to_dict())
for c in ["under", "sb", "sbu", "eb", "ebu", "kind"]:
    rg[c] = rg[c].astype(int)
print(f"도로명범위 {rg_all:,d}행 중 대상 지역 {len(rg):,d}행")

jr = pd.read_csv(RAW / "rangeaddr" / "20261013_지번범위.txt", sep="|", encoding="utf-8-sig", dtype=str, keep_default_na=False)
jr.columns = ["zip", "sido", "sido_en", "sgg", "sgg_en", "emd", "emd_en", "ri", "san", "hdong", "sb", "sbu", "eb", "ebu"]
jr_all = len(jr)
jr["code"] = [KEY2CODE.get(k) for k in zip(jr["sido"], jr["sgg"])]
jr = jr[jr["code"].notna()].copy()
jr_blank_end = int((jr["eb"] == "").sum())
jr["eb"] = np.where(jr["eb"] == "", jr["sb"], jr["eb"]); jr["ebu"] = np.where(jr["ebu"] == "", jr["sbu"], jr["ebu"])
print("지번범위 끝번호 빈 줄", jr_blank_end)
for c in ["san", "sb", "sbu", "eb", "ebu"]:
    jr[c] = jr[c].astype(int)
print(f"지번범위 {jr_all:,d}행 중 대상 지역 {len(jr):,d}행")

# 지번범위의 (읍면동, 리명)을 DB의 법정동 코드에 연결한다.
db["jkey_emd"] = np.where(db["읍면"] != "", db["읍면"], db["법정동명"])
area_tbl = db.drop_duplicates(["code", "법정동코드"])[["code", "법정동코드", "jkey_emd", "리명"]]
JKEY2BJD = collections.defaultdict(set)
for code, bjd, e, ri in area_tbl.itertuples(index=False):
    JKEY2BJD[(code, e, ri)].add(bjd)
jr["bjds"] = [JKEY2BJD.get(k) for k in zip(jr["code"], jr["emd"], jr["ri"])]
jr_unmatched = jr[jr["bjds"].isna()]
print(f"지번범위 법정동 연결 실패 {len(jr_unmatched):,d}행 ({len(jr_unmatched)/len(jr):.2%})")

도로명범위 끝번호 빈 줄 2582 / 범위종류별 {'3': 8282, '2': 3690, '1': 3658, '0': 2582}
도로명범위 312,087행 중 대상 지역 18,212행


지번범위 끝번호 빈 줄 3806
지번범위 253,726행 중 대상 지역 12,860행
지번범위 법정동 연결 실패 0행 (0.00%)


## 3. 시군구별 조각 만들기
숫자는 모두 정수 배열로 납작하게 펼쳐 크기를 줄인다. 형식 (자세한 내용은 `data/README.md`):
- `zips` 우편번호 목록(이하 번호는 이 목록의 위치), `areas` `[법정동코드, 이름, 읍면]`
- `roads[도로명].b` : `본번, 부번, 지하, 우편번호, 지역` 5개씩 / `roads[도로명].r` : `시작본, 시작부, 끝본, 끝부, 지하, 종류, 우편번호, 읍면` 8개씩
- `bn` 건물명 `[이름, 도로명, 본번, 부번, 우편번호, 지역]`
- `j[지역]` : `산, 본번, 부번, 우편번호` 4개씩 / `jr[지역]` : `산, 시작본, 시작부, 끝본, 끝부, 우편번호` 6개씩

In [4]:
def flat(rows):
    return [int(v) for row in rows for v in row]

stats = []
dup_dropped = 0
for code, g in db.groupby("code"):
    zips = sorted(set(g["우편번호"]) | set(rg.loc[rg["code"] == code, "zip"]) | set(jr.loc[jr["code"] == code, "zip"]))
    zi = {z: i for i, z in enumerate(zips)}
    areas = g.drop_duplicates("법정동코드").sort_values("법정동코드")[["법정동코드", "label", "emd"]]
    ai = {b: i for i, b in enumerate(areas["법정동코드"])}
    emds = sorted(set(areas["emd"]) | set(rg.loc[rg["code"] == code, "emd"]))
    ei = {e: i for i, e in enumerate(emds)}

    roads = {}
    bld = g[["도로명", "건물번호본번", "건물번호부번", "지하여부", "우편번호", "법정동코드"]].drop_duplicates()
    dup_dropped += len(g) - len(bld)
    for road, h in bld.sort_values(["도로명", "지하여부", "건물번호본번", "건물번호부번"]).groupby("도로명", sort=True):
        roads.setdefault(road, {})["b"] = flat(zip(h["건물번호본번"], h["건물번호부번"], h["지하여부"], h["우편번호"].map(zi), h["법정동코드"].map(ai)))
    for road, h in rg[rg["code"] == code].groupby("road", sort=True):
        roads.setdefault(road, {})["r"] = flat(zip(h["sb"], h["sbu"], h["eb"], h["ebu"], h["under"], h["kind"], h["zip"].map(zi), h["emd"].map(ei)))

    bn = g[g["시군구용건물명"] != ""][["시군구용건물명", "도로명", "건물번호본번", "건물번호부번", "우편번호", "법정동코드"]].drop_duplicates()
    bn_rows = [[n, r, int(b), int(u), zi[z], ai[a]] for n, r, b, u, z, a in bn.itertuples(index=False)]

    jb = g[["법정동코드", "산여부", "지번본번", "지번부번", "우편번호"]].drop_duplicates().sort_values(["법정동코드", "산여부", "지번본번", "지번부번"])
    j = {str(ai[b]): flat(zip(h["산여부"], h["지번본번"], h["지번부번"], h["우편번호"].map(zi))) for b, h in jb.groupby("법정동코드")}
    jrr = {}
    for _, row in jr[(jr["code"] == code) & jr["bjds"].notna()].iterrows():
        for b in row["bjds"]:
            jrr.setdefault(str(ai[b]), []).extend([row["san"], row["sb"], row["sbu"], row["eb"], row["ebu"], zi[row["zip"]]])

    shard = {"v": 1, "code": code, "name": CODE_NAME[code], "sido": CODE_SIDO[code], "base": BASE_DATE,
             "zips": zips, "emds": emds, "areas": areas.values.tolist(), "roads": roads, "bn": bn_rows, "j": j, "jr": jrr}
    p = ADDR / f"{code}.json"
    p.write_text(json.dumps(shard, ensure_ascii=False, separators=(",", ":"), allow_nan=False), encoding="utf-8")
    stats.append({"code": code, "name": CODE_NAME[code], "건물행": len(g), "도로수": len(roads), "건물명": len(bn_rows),
                  "지번 지역": len(j), "KB": round(p.stat().st_size / 1024, 1)})
st = pd.DataFrame(stats)
print(st.to_string(index=False)); print("합계 KB", round(st["KB"].sum()), " / 중복 제거된 건물행", dup_dropped)

 code    name   건물행  도로수  건물명  지번 지역     KB
30110      동구 25669  638 1597     44  773.1
30140      중구 25337  665 2053     26  810.2
30170      서구 24964  698 2980     27  817.8
30200     유성구 18253  712 2323     53  639.8
30230     대덕구 17883  566 2151     24  588.0
36110 세종특별자치시 28029 1135 1981    140  831.5
41111 수원시 장안구 13620  440 3391     11  520.1
41113 수원시 권선구 16983  576 2695     16  606.7
41115 수원시 팔달구 13781  444 2046     22  468.5
41117 수원시 영통구  6611  270 1329      7  248.3
41131 성남시 수정구 21289  608 1823     17  644.0
41133 성남시 중원구 12574  417 2536      9  457.2
41135 성남시 분당구  8529  523 1727     18  330.7
41461 용인시 처인구 42049 1169 7039     82 1434.5
41463 용인시 기흥구 13397  402 3117     18  524.6
41465 용인시 수지구  7106  230 1507      7  256.3
합계 KB 9951  / 중복 제거된 건물행 821


## 4. 이름 색인 (names.json)
입력에서 지역을 알아내는 용도다. 시군구 이름, 도로명, 읍면동·리 이름이 어느 시군구에 있는지 적는다.

In [5]:
codes = sorted(CODE_NAME)
cidx = {c: i for i, c in enumerate(codes)}
road_map, area_map = collections.defaultdict(set), collections.defaultdict(set)
for c in codes:
    sh = json.loads((ADDR / f"{c}.json").read_text(encoding="utf-8"))
    for r in sh["roads"]:
        road_map[r].add(cidx[c])
    for _, label, emd in sh["areas"]:
        for t in set(label.split()) | {label}:
            area_map[t].add(cidx[c])
names = {"v": 1, "c": [[c, CODE_NAME[c], CODE_SIDO[c]] for c in codes],
         "r": {k: sorted(v) for k, v in sorted(road_map.items())},
         "a": {k: sorted(v) for k, v in sorted(area_map.items())}}
(ADDR / "names.json").write_text(json.dumps(names, ensure_ascii=False, separators=(",", ":"), allow_nan=False), encoding="utf-8")
print("names.json KB", round((ADDR / "names.json").stat().st_size / 1024, 1), "/ 도로명", len(names["r"]), "/ 지역 이름", len(names["a"]))

names.json KB 232.8 / 도로명 9257 / 지역 이름 699


## 5. 전수 대조 검증
만든 조각 파일을 **다시 읽어** 조회 함수(사이트의 JavaScript가 그대로 따르는 규칙)로 DB의 모든 건물을 조회한다.
- 도로명 주소: 원래 우편번호가 `건물 일치`로 나오는지
- 지번 주소: 원래 우편번호가 결과에 들어 있는지 (한 필지에 우편번호가 둘 이상이면 모두 나온다)
- 범위 기준: 범위만으로 조회했을 때 DB 우편번호와 얼마나 같은지 (범위주소가 얼마나 믿을 만한지 가늠)

In [6]:
def road_lookup(sh, road, bon, bu, under, area_emd=None):
    """returns (level, [zips])  level: 'exact' | 'range' | None"""
    r = sh["roads"].get(road)
    if not r:
        return None, []
    b = r.get("b", [])
    hit = sorted({sh["zips"][b[i + 3]] for i in range(0, len(b), 5) if b[i] == bon and b[i + 1] == bu and b[i + 2] == under})
    if hit:
        return "exact", hit
    rr = r.get("r", [])
    zs = set()
    for i in range(0, len(rr), 8):
        sb, sbu, eb, ebu, u, kind, z, e = rr[i:i + 8]
        if u != under:
            continue
        if area_emd is not None and sh["emds"][e] != area_emd:
            continue
        if kind == 1 and bon % 2 == 0: continue
        if kind == 2 and bon % 2 == 1: continue
        if (sb, sbu) <= (bon, bu) <= (eb, ebu):
            zs.add(sh["zips"][z])
    return ("range", sorted(zs)) if zs else (None, [])

def jibun_lookup(sh, area_i, san, bon, bu):
    j = sh["j"].get(str(area_i), [])
    hit = sorted({sh["zips"][j[i + 3]] for i in range(0, len(j), 4) if j[i] == san and j[i + 1] == bon and j[i + 2] == bu})
    if hit:
        return "exact", hit
    jr_ = sh["jr"].get(str(area_i), [])
    zs = {sh["zips"][jr_[i + 5]] for i in range(0, len(jr_), 6) if jr_[i] == san and (jr_[i + 1], jr_[i + 2]) <= (bon, bu) <= (jr_[i + 3], jr_[i + 4])}
    return ("range", sorted(zs)) if zs else (None, [])

shards = {c: json.loads((ADDR / f"{c}.json").read_text(encoding="utf-8")) for c in codes}
res = collections.Counter(); bad_road, bad_range, multi_j = [], [], []
AI = {c: {a[0]: i for i, a in enumerate(shards[c]["areas"])} for c in codes}
RANGE_ONLY = {c: {**shards[c], "roads": {k: {"r": v.get("r", [])} for k, v in shards[c]["roads"].items()}} for c in codes}
for row in db.itertuples(index=False):
    sh = shards[row.code]
    lv, zs = road_lookup(sh, row.도로명, row.건물번호본번, row.건물번호부번, row.지하여부)
    if lv == "exact" and row.우편번호 in zs:
        res["도로명 건물 일치"] += 1
        if len(zs) > 1: res["도로명 일치 중 우편번호 둘 이상"] += 1
    else:
        res["도로명 실패"] += 1; bad_road.append((row.code, row.도로명, row.건물번호본번, row.건물번호부번, row.우편번호, str(zs)))
    # 범위만으로
    lvr, zr = road_lookup(RANGE_ONLY[row.code], row.도로명, row.건물번호본번, row.건물번호부번, row.지하여부, row.emd)
    if lvr is None:
        res["범위: 해당 범위 없음"] += 1
    elif zr == [row.우편번호]:
        res["범위: 같은 우편번호"] += 1
    elif row.우편번호 in zr:
        res["범위: 후보 여럿(정답 포함)"] += 1
    else:
        res["범위: 다른 우편번호"] += 1; bad_range.append((row.code, row.도로명, row.건물번호본번, row.건물번호부번, row.우편번호, str(zr)))
    lj, zj = jibun_lookup(sh, AI[row.code][row.법정동코드], row.산여부, row.지번본번, row.지번부번)
    if lj == "exact" and row.우편번호 in zj:
        res["지번 건물 일치"] += 1
        if len(zj) > 1: multi_j.append((row.code, row.법정동코드, row.산여부, row.지번본번, row.지번부번, ",".join(zj)))
    else:
        res["지번 실패"] += 1
n = len(db)
summary = pd.DataFrame([{"항목": k, "건수": v, "비율": round(v / n, 5)} for k, v in sorted(res.items())])
print(summary.to_string(index=False))
assert res["도로명 실패"] == 0, "도로명 전수 대조 실패 있음"
assert res["지번 실패"] == 0, "지번 전수 대조 실패 있음"

                항목     건수      비율
         도로명 건물 일치 296074 1.00000
도로명 일치 중 우편번호 둘 이상     26 0.00009
       범위: 같은 우편번호 296073 1.00000
  범위: 후보 여럿(정답 포함)      1 0.00000
          지번 건물 일치 296074 1.00000


## 6. 점검 엑셀 저장
한 파일에 시트별로 담는다. 범위주소와 건물 DB가 어긋나는 곳은 우체국 자료 자체의 한계이므로 지우지 않고 목록으로 남긴다.

In [7]:
cols = ["시군구코드", "도로명", "본번", "부번", "DB 우편번호", "조회 결과"]
limits = pd.DataFrame({"한계": [
    "범위주소(도로명범위·지번범위)는 우체국이 '참고자료'로 제공한다. 건물 DB와 어긋나는 곳이 있어 `범위 기준` 등급으로만 보여 준다.",
    "건물 DB에 없는 건물(신축·미등록)은 범위로만 조회된다. 범위도 없으면 '찾지 못함'이다.",
    "전수 대조에서 범위주소가 건물 DB와 거의 모두 같게 나오는 것은 두 자료가 같은 원천에서 만들어졌기 때문이며, 우편번호 자체가 현실과 맞는지를 독립적으로 검증한 결과는 아니다.",
    "전수 대조(도로명·지번 100%)는 '조각 파일을 만들고 다시 읽는 과정에서 값이 바뀌지 않았다'는 확인이다. 원본 DB의 정확성은 별개다.",
    "한 필지에 우편번호가 둘 이상인 지번은 모두 보여 준다 (지번 다중 우편번호 시트).",
    "건물명 검색은 DB의 '시군구용건물명'만 쓴다. 다량배달처명은 이 DB에서 모두 비어 있다.",
    "지번범위를 법정동에 연결하지 못한 행은 범위 기준 조회에서 빠진다 (지번범위 연결 실패 시트).",
    "이름 색인(names.json)은 선택 지역만 담고 있다. 전국 규모에서는 크기를 보고 분할 방식을 다시 정한다.",
    f"기준일 {BASE_DATE}, 고시일 {NOTICE_DATE}. 이후 변경분(변경분 DB)은 반영하지 않았다.",
]})
xl = OUT / "qa" / "address_index_qa_v1.xlsx"
xl.parent.mkdir(exist_ok=True, parents=True)
with pd.ExcelWriter(xl, engine="openpyxl") as w:
    summary.to_excel(w, sheet_name="전수대조_요약", index=False)
    st.to_excel(w, sheet_name="시군구별_조각", index=False)
    pd.DataFrame(bad_range[:5000], columns=cols).to_excel(w, sheet_name="범위_다른우편번호", index=False)
    pd.DataFrame(multi_j, columns=["시군구코드", "법정동코드", "산", "본번", "부번", "우편번호들"]).to_excel(w, sheet_name="지번_다중우편번호", index=False)
    jr_unmatched.drop(columns=["bjds"]).head(5000).to_excel(w, sheet_name="지번범위_연결실패", index=False)
    limits.to_excel(w, sheet_name="데이터_한계", index=False)
meta = {"built_from": {"base_date": BASE_DATE, "notice_date": NOTICE_DATE}, "sigungu": len(codes), "building_rows": int(n),
        "road_names": len(names["r"]), "shard_kb_total": round(float(st["KB"].sum()), 1),
        "results": {k: int(v) for k, v in res.items()}, "jibun_range_unmatched_rows": int(len(jr_unmatched))}
(ADDR / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(meta, ensure_ascii=False, indent=1))

{
 "built_from": {
  "base_date": "2026-10-08",
  "notice_date": "2026-10-13"
 },
 "sigungu": 16,
 "building_rows": 296074,
 "road_names": 9257,
 "shard_kb_total": 9951.3,
 "results": {
  "도로명 건물 일치": 296074,
  "범위: 같은 우편번호": 296073,
  "지번 건물 일치": 296074,
  "도로명 일치 중 우편번호 둘 이상": 26,
  "범위: 후보 여럿(정답 포함)": 1
 },
 "jibun_range_unmatched_rows": 0
}
